# EMiF Project Final - Has the structure of risk in financial markets changed since COVID-19?

This notebook is the final synthesis notebook for the project. It does not re-estimate every model from scratch. Instead, it reloads the key outputs produced by notebooks 01 to 05 and organizes them into one coherent empirical answer to the research question.

The logic of the project is cumulative:
1. describe the transformed data and stylized facts;
2. test whether volatility dynamics changed;
3. test whether cross-market dependence changed;
4. test whether transmission channels changed;
5. test whether regimes and structural breaks changed.

The final conclusion must therefore come from the joint reading of all five empirical blocks rather than from one single model.

## 1. Setup

We keep the final notebook intentionally light. The heavy estimation work is done in the five analytical notebooks; here we simply read their outputs and synthesize the economic message.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path.cwd().resolve()
TABLE_DIR = ROOT / "outputs" / "project2" / "tables"
FIGURE_DIR = ROOT / "outputs" / "project2" / "figures"

pd.options.display.float_format = "{:.4f}".format

## 2. Research question and empirical design

The project asks whether the **structure of risk** changed after COVID-19. In this context, ?structure of risk? means more than just higher volatility. It includes:
- the distribution of returns and downside risk;
- conditional volatility persistence and asymmetry;
- cross-asset dependence;
- transmission across markets;
- latent regimes and structural breaks.

This definition motivates the sequence of methods used throughout the project.

## 3. Stylized facts and break-date justification

We start with descriptive evidence. If the post-COVID period really differs from the earlier sample, the first signs should appear in moments, tail risk, correlation patterns and the variance-break tests on the S&P 500.

In [ ]:
stylized_facts = pd.read_csv(TABLE_DIR / "01_stylized_facts_pre_post.csv")
chow_break = pd.read_csv(TABLE_DIR / "01_chow_variance_break_at_covid.csv")
quandt_breaks = pd.read_csv(TABLE_DIR / "01_quandt_top_breaks.csv")

print("Stylized facts for the five core markets used later in the project")
display(stylized_facts.loc[stylized_facts["asset"].isin(["sp500", "ust10y_yield", "oil", "eurusd", "us_hy_bonds"])])

print("Variance break test imposed at the COVID date")
display(chow_break)

print("Top Quandt-Andrews style variance break candidates")
display(quandt_breaks)

In [ ]:
display(Image(filename=str(FIGURE_DIR / "01_pre_post_correlation_heatmaps.png")))
display(Image(filename=str(FIGURE_DIR / "01_sp500_variance_breaks.png")))

## 4. Univariate volatility: did persistence or asymmetry change?

The next question is whether post-COVID risk became more persistent or more asymmetric at the single-asset level. This is exactly what the GARCH, GJR-GARCH and GARCH-t models test.

In [ ]:
garch_summary = pd.read_csv(TABLE_DIR / "02_garch_summary.csv")

print("GARCH persistence and diagnostics")
display(garch_summary[[
    "asset", "sample", "model", "alpha_1", "beta_1", "gamma_1", "nu", "persistence",
    "lb_resid_pvalue_lag10", "lb_sq_resid_pvalue_lag10", "engle_ng_joint_pvalue"
]])

In [ ]:
display(Image(filename=str(FIGURE_DIR / "02_conditional_volatility_sp500.png")))
display(Image(filename=str(FIGURE_DIR / "02_conditional_volatility_us_hy_bonds.png")))

## 5. Dependence and co-movement after COVID

A change in the structure of risk should also appear in cross-asset co-movements. We therefore look at rolling correlations and then at the DCC-GARCH estimate for the key SPX / US 10Y pair.

In [ ]:
rolling_corr_summary = pd.read_csv(TABLE_DIR / "03_rolling_correlation_summary.csv")
dcc_summary = pd.read_csv(TABLE_DIR / "03_dcc_summary.csv")
forbes_rigobon = pd.read_csv(TABLE_DIR / "03_forbes_rigobon_summary.csv")

print("Rolling correlation summary")
display(rolling_corr_summary)

print("DCC summary for S&P 500 vs US 10Y yield change")
display(dcc_summary)

print("Forbes-Rigobon adjustment")
display(forbes_rigobon)

In [ ]:
display(Image(filename=str(FIGURE_DIR / "03_rolling_correlations.png")))
display(Image(filename=str(FIGURE_DIR / "03_spx_ust_dcc_path.png")))

## 6. Transmission and spillovers across markets

Even if dependence changes, that does not necessarily mean transmission changes. The VAR(2), Granger tests and Geweke measures address exactly this point by studying whether shocks and forecast content move differently across markets after COVID.

In [ ]:
var_summary = pd.read_csv(TABLE_DIR / "04_var_summary.csv")
granger_results = pd.read_csv(TABLE_DIR / "04_granger_results.csv")
geweke_results = pd.read_csv(TABLE_DIR / "04_geweke_results.csv")

print("VAR summary")
display(var_summary)

print("Granger links with p-value below 5%")
display(granger_results.loc[granger_results["granger_pvalue"] < 0.05])

print("Geweke directional causality with p-value below 5%")
display(geweke_results.loc[(geweke_results["measure"] == "directional") & (geweke_results["pvalue"] < 0.05)])

print("Geweke instantaneous causality with p-value below 5%")
display(geweke_results.loc[(geweke_results["measure"] == "instantaneous") & (geweke_results["pvalue"] < 0.05)])

In [ ]:
display(Image(filename=str(FIGURE_DIR / "04_irf_pre_covid.png")))
display(Image(filename=str(FIGURE_DIR / "04_irf_post_covid.png")))

## 7. Regimes and structural breaks

This is the strongest block of evidence in the whole project. A latent-regime model tells us whether the market alternates between different dependence states, while the break detection procedure tells us whether the underlying risk process changed discretely at identifiable dates.

In [ ]:
markov_summary = pd.read_csv(TABLE_DIR / "05_markov_summary.csv")
variance_breaks = pd.read_csv(TABLE_DIR / "05_sp500_variance_breaks.csv")
correlation_breaks = pd.read_csv(TABLE_DIR / "05_spx_ust_corr_breaks.csv")

print("Markov-switching summary")
display(markov_summary)

print("Variance breaks in the S&P 500 rolling variance")
display(variance_breaks)

print("Breaks in the SPX / UST rolling correlation")
display(correlation_breaks)

In [ ]:
display(Image(filename=str(FIGURE_DIR / "05_markov_smoothed_probabilities.png")))
display(Image(filename=str(FIGURE_DIR / "05_sp500_variance_breaks.png")))
display(Image(filename=str(FIGURE_DIR / "05_spx_ust_corr_breaks.png")))

## 8. Final answer to the research question

Taken together, the evidence points to a **meaningful post-COVID change in the structure of risk**, but not in a one-dimensional sense.

The most convincing elements are:
- the clear change in rolling correlation patterns, especially for the SPX / US 10Y pair;
- the high persistence of the DCC correlation process;
- the significant Granger and Geweke transmission links in the VAR system;
- the Markov-switching evidence of distinct correlation regimes;
- the structural breaks detected in both the S&P 500 variance and the SPX / UST dependence process.

At the same time, the project also suggests nuance. The post-COVID change is not simply ?more risk everywhere?. Instead, it looks like a **reorganization of how risk is priced, transmitted and shared across markets**.

## 9. Economic interpretation

A natural reading of the results is that COVID did not just create a temporary volatility shock. It seems to have changed the interaction between growth-sensitive assets, sovereign yields and credit markets in a more persistent way. In particular, the equity/rates relationship appears less stable after COVID, while transmission and regime evidence suggest that market stress now propagates through a different dependence structure than before 2020.

## 10. Limits and next steps

This project remains intentionally within the scope of course econometrics. The results are reduced-form and depend on the chosen sample split, rolling windows and bivariate/trivariate summaries. Still, within those limits, the combined evidence is internally coherent and supports the idea that the structure of risk changed after COVID-19.